In [1]:
import ast
import json
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split, GridSearchCV
import fasttext
import fasttext.util
from itertools import product

from UtilsRF import scorer

Downloading FastText takes a considerable amount of time, so we have decided to provide the user with the pre-downloaded and reduced model. However, since it is a large file, it is not feasible to upload it to GitHub. Therefore, we have uploaded it to the following One Drive folder: https://tuwienacat-my.sharepoint.com/:f:/r/personal/e12302519_student_tuwien_ac_at/Documents/NLP/Shared?csf=1&web=1&e=cRG8yN . Before running the notebook, it is important to download the file cc.en.300_reduced.bin from that folder and save it in the same directory as the notebook (docs/final_submission/RF).

 # LOAD FASTTEXT
 load the reduced version with 100 dimensions.

In [2]:
# Load the reduced model
ft = fasttext.load_model("cc.en.300_reduced.bin")

# LOAD THE FILE AND CREATE A DATAFRAME

In [11]:
with open("../../../data/tacred/json/train.json", "r") as file:
    data_train = json.load(file)

with open("../../../data/tacrev/json/test.json", "r") as file:
    data_test = json.load(file)

with open("../../../data/tacrev/json/dev.json", "r") as file:
    data_dev = json.load(file)

In [12]:
# Create a DataFrame from the loaded data
df_test = pd.DataFrame(data_test)
df_train = pd.DataFrame(data_train)
df_dev = pd.DataFrame(data_dev)

# EMBEDDINGS CREATION

Creation of embeddingd for each token using FastText pretrained model reduced to 100 dimensions 

In [13]:
# Generate embeddings for each token in the list
def get_embeddings(tokens):
    embeddings = []
    max_length = max(len(token_list) for token_list in tokens)

    for token_list in tokens:
        token_embeddings = [ft.get_word_vector(token) for token in token_list]
        # Padding or truncating embeddings to match the maximum length
        padding = [np.zeros_like(token_embeddings[0])] * (max_length - len(token_list))
        padded_embeddings = (
            token_embeddings + padding
            if len(token_list) < max_length
            else token_embeddings[:max_length]
        )
        embeddings.append(padded_embeddings)

    return embeddings

In [14]:
df_train["embeddings"] = get_embeddings(df_train["token"])
df_train["mean_embeddings"] = df_train["embeddings"].apply(np.mean)

In [15]:
df_test["embeddings"] = get_embeddings(df_test["token"])
df_test["mean_embeddings"] = df_test["embeddings"].apply(np.mean)

In [16]:
df_dev["embeddings"] = get_embeddings(df_dev["token"])
df_dev["mean_embeddings"] = df_dev["embeddings"].apply(np.mean)

# COMPUTE THE DISTANCE BETWEEN SUBJ AND OBJ

As additional feature, it computes the distance between subject and object

In [17]:
# Calculate the distance between subject and object
def calculate_distance(df):
    df["subj_midpoint"] = (df["subj_start"] + df["subj_end"]) / 2
    df["obj_midpoint"] = (df["obj_start"] + df["obj_end"]) / 2
    df["subj_obj_distance"] = df["obj_midpoint"] - df["subj_midpoint"]
    df = df.drop(columns=["subj_midpoint", "obj_midpoint"])
    return df

In [18]:
df_train = calculate_distance(df_train)

In [19]:
df_test = calculate_distance(df_test)

In [20]:
df_dev = calculate_distance(df_dev)

# COMPUTE COSINE SIMILARITY BETWEEN SUBJ AND OBJ

Creation of a column for subject and one for object in order to compute the cosine similarity.

In [21]:
def extract_subject_object(row):
    subj_start = row["subj_start"]
    subj_end = row["subj_end"]
    obj_start = row["obj_start"]
    obj_end = row["obj_end"]

    subject = " ".join(row["token"][subj_start : subj_end + 1])
    object_ = " ".join(row["token"][obj_start : obj_end + 1])

    return pd.Series({"Subject": subject, "Object": object_})

In [22]:
df_train[["Subject", "Object"]] = df_train.apply(extract_subject_object, axis=1)

In [23]:
df_test[["Subject", "Object"]] = df_test.apply(extract_subject_object, axis=1)

In [24]:
df_dev[["Subject", "Object"]] = df_dev.apply(extract_subject_object, axis=1)

In [25]:
# Compute cosine similarity between corresponding pairs of subject and object embeddings
subject_embeddings = [ft.get_word_vector(word) for word in df_train["Subject"]]
object_embeddings = [ft.get_word_vector(word) for word in df_train["Object"]]

similarities = [
    cosine_similarity([sub_emb], [obj_emb])[0][0]
    for sub_emb, obj_emb in zip(subject_embeddings, object_embeddings)
]

df_train["word_similarity"] = similarities

In [26]:
# Compute cosine similarity between corresponding pairs of subject and object embeddings
subject_embeddings = [ft.get_word_vector(word) for word in df_test["Subject"]]
object_embeddings = [ft.get_word_vector(word) for word in df_test["Object"]]

similarities = [
    cosine_similarity([sub_emb], [obj_emb])[0][0]
    for sub_emb, obj_emb in zip(subject_embeddings, object_embeddings)
]

df_test["word_similarity"] = similarities

In [27]:
# Compute cosine similarity between corresponding pairs of subject and object embeddings
subject_embeddings = [ft.get_word_vector(word) for word in df_dev["Subject"]]
object_embeddings = [ft.get_word_vector(word) for word in df_dev["Object"]]

similarities = [
    cosine_similarity([sub_emb], [obj_emb])[0][0]
    for sub_emb, obj_emb in zip(subject_embeddings, object_embeddings)
]

df_dev["word_similarity"] = similarities

# ONE HOT ENCODING OF OBJ_TYPES AND SUBJ_TYPES

Creation of a column for each subject and object type and performs one-hot encoding on them, creating binary columns to represent unique categories within each specified column

In [28]:
# Add additional binary columns for each unique category in the specified columns
def perform_one_hot_encoding(df, columns=["subj_type", "obj_type"]):
    for column in columns:
        dummies = pd.get_dummies(df[column], prefix=column)
        df = pd.concat([df, dummies], axis=1)
    return df

In [29]:
df_train = perform_one_hot_encoding(df_train, columns=["subj_type", "obj_type"])

In [30]:
df_test = perform_one_hot_encoding(df_test, columns=["subj_type", "obj_type"])

In [31]:
df_dev = perform_one_hot_encoding(df_dev, columns=["subj_type", "obj_type"])

# ONE HOT ENCODING OF STANDFORD POS

This function conducts one-hot encoding on stanford_POS column that contains lists of tags. It creates binary columns for each unique tag value, assigning 1 to cells where a tag is present and 0 otherwise. The resulting DataFrame incorporates the new one-hot encoded columns for each unique tag value.

In [32]:
def one_hot_encoding_pos(df, column_name):
    unique_tags = set(tag for tag_list in df[column_name] for tag in tag_list)

    for tag_value in unique_tags:
        df[tag_value] = 0

    for index, row in df.iterrows():
        for tag in row[column_name]:
            df.at[index, tag] = 1

    return df

In [33]:
df_train = one_hot_encoding_pos(df_train, "stanford_pos")

In [34]:
df_test = one_hot_encoding_pos(df_test, "stanford_pos")

In [35]:
df_dev = one_hot_encoding_pos(df_dev, "stanford_pos")

# DELETE THE USELESS COLUMNS FOR THE MODEL

In [36]:
df_train = df_train.drop(
    columns=[
        "id",
        "docid",
        "embeddings",
        "token",
        "subj_start",
        "subj_end",
        "obj_start",
        "obj_end",
        "subj_type",
        "obj_type",
        "stanford_ner",
        "stanford_pos",
        "stanford_head",
        "stanford_deprel",
        "Subject",
        "Object",
    ]
)

In [37]:
df_test = df_test.drop(
    columns=[
        "id",
        "docid",
        "embeddings",
        "token",
        "subj_start",
        "subj_end",
        "obj_start",
        "obj_end",
        "subj_type",
        "obj_type",
        "stanford_ner",
        "stanford_pos",
        "stanford_head",
        "stanford_deprel",
        "Subject",
        "Object",
    ]
)

In [38]:
df_dev = df_dev.drop(
    columns=[
        "id",
        "docid",
        "embeddings",
        "token",
        "subj_start",
        "subj_end",
        "obj_start",
        "obj_end",
        "subj_type",
        "obj_type",
        "stanford_ner",
        "stanford_pos",
        "stanford_head",
        "stanford_deprel",
        "Subject",
        "Object",
    ]
)

# RANDOM FOREST CLASSIFIER

Prepare data to be given as input to the Random Forest Classieier and then try different classifiers with different combinations of values for number of estimators and for max depth. It saves the parameters with which it achieves the best result and then train a new classifier with that parameter.The dataset is balanced thanks to class_weight parameter of RandomForestClassifier.

In [39]:
X_train = df_train.drop(columns=["relation"])
y_train = df_train["relation"]

X_test = df_test.drop(columns=["relation"])
y_test = df_test["relation"]

X_dev = df_dev.drop(columns=["relation"])
y_dev = df_dev["relation"]

In [40]:
n_estimators_values = [100, 200, 300]
max_depth_values = [10, 20, 50]
best_f1_micro = 0
dev = y_dev.tolist()

# Iterate over all combinations of hyperparameter values
for n_estimators, max_depth in product(n_estimators_values, max_depth_values):
    rf_classifier = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight="balanced",
        random_state=42,
    )

    rf_classifier.fit(X_train, y_train)
    predictions = rf_classifier.predict(X_dev)
    predictions = predictions.tolist()

    prec_micro, recall_micro, current_f1_micro = scorer.score(
        y_dev, predictions, verbose=False
    )

    if current_f1_micro > best_f1_micro:
        best_f1_micro = current_f1_micro
        best_params = {
            "n_estimators_values": n_estimators,
            "max_depth_values": max_depth,
        }

# Create the Random Forest classifier with the best parameters
best_rf_classifier = RandomForestClassifier(
    n_estimators=best_params["n_estimators_values"],
    max_depth=best_params["max_depth_values"],
    class_weight="balanced",
    random_state=42,
)

best_rf_classifier.fit(X_train, y_train)

c:\Users\Hp\anaconda3\envs\tuwnlpie\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


Precision (micro): 19.562%
   Recall (micro): 77.547%
       F1 (micro): 31.243%


c:\Users\Hp\anaconda3\envs\tuwnlpie\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


Precision (micro): 49.806%
   Recall (micro): 48.396%
       F1 (micro): 49.091%


c:\Users\Hp\anaconda3\envs\tuwnlpie\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


...

Precision (micro): 19.105%
   Recall (micro): 77.642%
       F1 (micro): 30.664%


c:\Users\Hp\anaconda3\envs\tuwnlpie\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


Precision (micro): 50.622%
   Recall (micro): 47.604%
       F1 (micro): 49.067%


c:\Users\Hp\anaconda3\envs\tuwnlpie\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


Precision (micro): 83.241%
   Recall (micro): 17.057%
       F1 (micro): 28.312%


In [41]:
best_predictions = best_rf_classifier.predict(X_test)

best_predictions = best_predictions.tolist()
y_test = y_test.tolist()

scorer.score(y_test, best_predictions, verbose=False)

c:\Users\Hp\anaconda3\envs\tuwnlpie\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


Precision (micro): 46.604%
   Recall (micro): 49.440%
       F1 (micro): 47.980%


(0.4660428614548747, 0.49439641370477105, 0.4798011187072716)

In [42]:
# Show parameters
params = best_rf_classifier.get_params()
for param, value in params.items():
    print(f"{param}: {value}")

bootstrap: True
ccp_alpha: 0.0
class_weight: balanced
criterion: gini
max_depth: 20
max_features: sqrt
max_leaf_nodes: None
max_samples: None
min_impurity_decrease: 0.0
min_samples_leaf: 1
min_samples_split: 2
min_weight_fraction_leaf: 0.0
n_estimators: 100
n_jobs: None
oob_score: False
random_state: 42
verbose: 0
warm_start: False


In [44]:
# Create a dataframe with columns "Gold Label" and "Prediction" and save it in a .txt
df_results_Tacrev = pd.DataFrame({"Gold Label": y_test, "Prediction": best_predictions})
df_results_Tacrev.to_csv("predictions_tacrev.txt", sep="\t", index=False)